# 📂 Notebook 1 — Exploração e Preparação de Dados

**Projeto**: Análise de Incidentes de IA em Serviços Financeiros  
**Metodologia**: CRISP-DM | **Fase**: Data Understanding + Data Preparation  
**Instituição**: PUC-SP

---

## 🎯 Contextualização

Este notebook implementa as fases de **Data Understanding** e **Data Preparation** da metodologia CRISP-DM. A base utilizada é a **AI Incident Database (AIID)**, consultada via endpoint GraphQL oficial com fallback automático para CSV local.

## 📋 O que este notebook entrega

1. ✅ Coleta via API GraphQL AIID com fallback CSV  
2. ✅ Exploração inicial e análise de qualidade  
3. ✅ Filtragem temática para o setor financeiro  
4. ✅ Limpeza e padronização  
5. ✅ Engenharia de 8 variáveis derivadas  
6. ✅ Visualização das distribuições  
7. ✅ `incidents_finance_filtered.csv` — dataset processado  
8. ✅ `ai_finance_incidents.db` — banco SQLite (3 tabelas relacionadas)

---

## 1️⃣ Instalação de dependências

**O que faz**: Instala bibliotecas que podem não estar no Google Colab por padrão.

**Por que é importante**: Garante ambiente reproduzível antes de qualquer importação.

In [1]:
!conda install -q -y -n <your_environment_name> xgboost shap flask flask-cors joblib

zsh:1: no such file or directory: your_environment_name


## 2️⃣ Importação de bibliotecas

**O que faz**: Importa todos os pacotes necessários para coleta, processamento e persistência dos dados.

**Bibliotecas principais**: `pandas` / `numpy` — dados | `sqlite3` — banco SQL | `requests` — chamadas HTTP | `matplotlib` / `seaborn` — visualização

In [2]:
import os, re, json, sqlite3, warnings, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1200)

print('='*80)
print('✅ BIBLIOTECAS IMPORTADAS COM SUCESSO')
print('='*80)
print(f'📌 Pandas : {pd.__version__}')
print(f'📌 NumPy  : {np.__version__}')
print('\n🚀 Ambiente configurado!')

✅ BIBLIOTECAS IMPORTADAS COM SUCESSO
📌 Pandas : 3.0.2
📌 NumPy  : 2.4.4

🚀 Ambiente configurado!


## 3️⃣ Carregamento dos Dados — API AIID com Fallback em 4 Camadas

**O que faz**: Obtém incidentes do AI Incident Database (AIID) via GraphQL.

**Estratégia de fallback (4 camadas):**
1. 🌐 **API GraphQL — endpoint `incidents`** (fonte primária correta)
2. 📦 **Cache JSON local** (evita re-chamadas desnecessárias)
3. 📄 **Cache CSV local** (snapshot da última chamada bem-sucedida)
4. 📁 **`incidents.csv` local** (Kaggle — garante execução offline)

**Bugs corrigidos nesta célula:**
- ⚠️ Query anterior buscava `reports` (artigos de notícia) em vez de `incidents` (eventos reais)
- ⚠️ Campos `description` e `date` não eram retornados pela query anterior → filtro financeiro falhava
- ⚠️ Normalize function mapeava campos errados (`text` → `description` em vez de usar `description` diretamente)
- ⚠️ Sem retry em timeout de rede
- ⚠️ Sem fallback para `incidents.csv` quando todos os outros falham

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURAÇÃO — caminhos e constantes
# ══════════════════════════════════════════════════════════════════════════════
AIID_URL   = "https://incidentdatabase.ai/api/graphql"
CACHE_JSON = "aiid_incidents_raw.json"   # cache da resposta bruta da API
CACHE_CSV  = "aiid_incidents_raw.csv"    # cache CSV da última chamada
LOCAL_CSV  = "incidents.csv"             # fallback Kaggle (upload manual)

# ── Query para INCIDENTS (eventos reais) ──────────────────────────────────────
# BUG ANTERIOR: a query buscava 'reports' (artigos de imprensa sobre incidentes)
# em vez de 'incidents' (os eventos em si). Reports têm report_number + text,
# enquanto incidents têm incident_id + description + date — campos necessários
# para o filtro financeiro e para o banco SQLite final.
GQL_INCIDENTS = """
query {
  incidents {
    incident_id
    title
    description
    date
  }
}
"""

# ── Query alternativa para REPORTS (caso endpoint incidents esteja indisponível)
# Retorna artigos de imprensa; menos rico, mas ainda utilizável como fallback.
GQL_REPORTS = """
query {
  reports {
    report_number
    title
    text
    date_published
    url
  }
}
"""

# ══════════════════════════════════════════════════════════════════════════════
# FUNÇÃO: normalizar resposta da API em DataFrame padronizado
# ══════════════════════════════════════════════════════════════════════════════
def normalize_incidents(records: list) -> pd.DataFrame:
    """
    Normaliza registros da query 'incidents' para DataFrame padronizado.

    A query incidents retorna: incident_id, title, description, date
    que já mapeiam 1:1 para as colunas que o pipeline usa.
    """
    df = pd.DataFrame(records)

    # Garantir que todas as colunas esperadas existem
    if "incident_id" not in df.columns:
        df["incident_id"] = range(1, len(df) + 1)
    if "title" not in df.columns:
        df["title"] = "Untitled"
    if "description" not in df.columns:
        df["description"] = ""
    if "date" not in df.columns:
        df["date"] = pd.NaT

    return df


def normalize_reports(records: list) -> pd.DataFrame:
    """
    Normaliza registros da query 'reports' para o mesmo formato padronizado.

    Reports retornam: report_number, title, text, date_published, url
    Precisamos renomear para corresponder ao schema de incidents.

    ATENÇÃO: 'text' aqui é o corpo do artigo (≠ 'description' do incident).
    O filtro financeiro funcionará, mas os IDs serão de reports, não incidents.
    """
    df = pd.DataFrame(records)

    rename = {
        "report_number" : "incident_id",
        "text"          : "description",   # corpo do artigo → description
        "date_published": "date",
    }
    for old, new in rename.items():
        if old in df.columns and new not in df.columns:
            df = df.rename(columns={old: new})

    if "incident_id" not in df.columns or df["incident_id"].isnull().all():
        df["incident_id"] = range(1, len(df) + 1)
    if "description" not in df.columns:
        df["description"] = ""
    if "date" not in df.columns:
        df["date"] = pd.NaT

    return df


def normalize_csv(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normaliza o CSV do Kaggle (incidents.csv) para o mesmo schema.

    O CSV tem colunas extras (deployer, developer, etc.) que são preservadas.
    Apenas garantimos que as colunas mínimas necessárias existam.
    """
    rename = {
        "date"         : "date",          # já correto
        "Alleged deployer of AI system"              : "deployer",
        "Alleged developer of AI system"             : "developer",
        "Alleged harmed or nearly harmed parties"    : "harmed_parties",
    }
    for old, new in rename.items():
        if old in df.columns and new not in df.columns:
            df = df.rename(columns={old: new})

    if "incident_id" not in df.columns:
        df["incident_id"] = range(1, len(df) + 1)
    if "description" not in df.columns:
        df["description"] = ""

    return df


# ══════════════════════════════════════════════════════════════════════════════
# CARREGAMENTO COM FALLBACK EM 4 CAMADAS
# ══════════════════════════════════════════════════════════════════════════════
df_original = None
source      = None

# ── Camada 1: API GraphQL — query incidents ───────────────────────────────────
try:
    print("🌐 Tentando API AIID — endpoint incidents...")
    r = requests.post(
        AIID_URL,
        json={"query": GQL_INCIDENTS},
        timeout=60,
        headers={"Content-Type": "application/json"},
    )
    r.raise_for_status()
    payload = r.json()

    if "errors" in payload:
        raise ValueError(f"Erros GraphQL: {payload['errors']}")

    records = payload.get("data", {}).get("incidents", [])

    if not records:
        # Alguns endpoints retornam incidents vazio — tentar reports
        print("  ⚠️  'incidents' vazio — tentando 'reports'...")
        r2 = requests.post(AIID_URL, json={"query": GQL_REPORTS}, timeout=60)
        r2.raise_for_status()
        payload2 = r2.json()
        records  = payload2.get("data", {}).get("reports", [])
        if not records:
            raise ValueError("API retornou zero registros em ambas as queries.")
        df_original = normalize_reports(records)
        source      = "API GraphQL AIID — reports"
    else:
        df_original = normalize_incidents(records)
        source      = "API GraphQL AIID — incidents"

    # Salvar cache para evitar chamadas repetidas
    with open(CACHE_JSON, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    df_original.to_csv(CACHE_CSV, index=False)
    print(f"  ✅ Cache salvo: {CACHE_JSON} e {CACHE_CSV}")

except Exception as e1:
    print(f"  ❌ API indisponível: {type(e1).__name__}: {e1}")

    # ── Camada 2: Cache JSON ──────────────────────────────────────────────────
    if df_original is None and os.path.exists(CACHE_JSON):
        try:
            print("\n📦 Carregando cache JSON local...")
            with open(CACHE_JSON, encoding="utf-8") as f:
                records = json.load(f)
            # Detectar se é incidents ou reports pelo campo presente
            if records and "incident_id" in records[0]:
                df_original = normalize_incidents(records)
            else:
                df_original = normalize_reports(records)
            source = "Cache JSON local"
            print(f"  ✅ {len(records):,} registros do cache JSON")
        except Exception as e2:
            print(f"  ❌ Cache JSON inválido: {e2}")

    # ── Camada 3: Cache CSV ───────────────────────────────────────────────────
    if df_original is None and os.path.exists(CACHE_CSV):
        try:
            print("\n📄 Carregando cache CSV local...")
            df_original = pd.read_csv(CACHE_CSV, low_memory=False)
            source      = "Cache CSV local"
            print(f"  ✅ {len(df_original):,} registros do cache CSV")
        except Exception as e3:
            print(f"  ❌ Cache CSV inválido: {e3}")

    # ── Camada 4: incidents.csv (Kaggle) ──────────────────────────────────────
    if df_original is None and os.path.exists(LOCAL_CSV):
        try:
            print("\n📁 Carregando incidents.csv local (Kaggle)...")
            df_raw      = pd.read_csv(LOCAL_CSV, low_memory=False)
            df_original = normalize_csv(df_raw)
            source      = "incidents.csv (Kaggle — upload local)"
            print(f"  ✅ {len(df_original):,} registros do CSV local")
        except Exception as e4:
            print(f"  ❌ incidents.csv inválido: {e4}")

# ── Verificação final ─────────────────────────────────────────────────────────
if df_original is None:
    raise FileNotFoundError(
        "\n❌ NENHUMA FONTE DE DADOS DISPONÍVEL.\n"
        "Opções para resolver:\n"
        "  1. Garantir acesso à internet para a API AIID\n"
        "  2. Fazer upload do arquivo 'incidents.csv' na pasta do projeto\n"
        "  3. Executar o notebook com o cache JSON/CSV de uma execução anterior"
    )

# ── Relatório de carregamento ─────────────────────────────────────────────────
print()
print("=" * 80)
print(f"📊 DATASET CARREGADO — {source.upper()}")
print("=" * 80)
print(f"✅ Registros  : {len(df_original):,}")
print(f"📋 Colunas    : {len(df_original.columns)}")
print(f"💾 Memória    : {df_original.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

if "date" in df_original.columns:
    datas = pd.to_datetime(df_original["date"], errors="coerce")
    if datas.notna().any():
        print(f"📅 Período    : {datas.min().year} → {datas.max().year}")

print(f"\n📌 Colunas disponíveis: {df_original.columns.tolist()}")
print()
display(df_original.head(3))

🌐 Tentando API AIID — endpoint incidents...
  ❌ API indisponível: HTTPError: 403 Client Error: Forbidden for url: https://incidentdatabase.ai/api/graphql


FileNotFoundError: 
❌ NENHUMA FONTE DE DADOS DISPONÍVEL.
Opções para resolver:
  1. Garantir acesso à internet para a API AIID
  2. Fazer upload do arquivo 'incidents.csv' na pasta do projeto
  3. Executar o notebook com o cache JSON/CSV de uma execução anterior

## 4️⃣ Exploração inicial e qualidade dos dados

**O que faz**: Inspeciona estrutura, tipos de dados, valores ausentes e duplicatas.

**Por que é importante**: Conhecer a qualidade antes da limpeza evita erros propagados.

In [ ]:
print("="*80); print("📋 ESTRUTURA DO DATASET"); print("="*80)
for i,col in enumerate(df_original.columns, 1):
    print(f"  {i:2d}. {col}")
df_original.info()

missing = pd.DataFrame({
    "Coluna"      : df_original.columns,
    "Ausentes"    : df_original.isnull().sum().values,
    "Percentual_%": (df_original.isnull().sum().values / len(df_original) * 100).round(2)
}).query("Ausentes > 0").sort_values("Ausentes", ascending=False)

print("\n" + "="*80); print("🔍 VALORES AUSENTES"); print("="*80)
print(missing.to_string(index=False) if len(missing) > 0 else "✅ Nenhum valor ausente.")

id_col = next((c for c in ["incident_id","report_number"] if c in df_original.columns), None)
if id_col:
    print(f"\n🔄 Duplicatas em [{id_col}]: {df_original.duplicated(subset=[id_col]).sum()}")

## 5️⃣ Filtragem temática — Setor financeiro

**O que faz**: Seleciona incidentes de bancos, fintechs, crédito, trading e subdomínios financeiros por palavras-chave.

**Caso de uso**: Reduzir o escopo ao setor de interesse, eliminando ruído de outros domínios.

In [ ]:
FINANCIAL_KEYWORDS = [
    "finance","financial","banking","bank","fintech",
    "credit","loan","mortgage","debt","lending",
    "fraud detection","anti-money laundering","aml",
    "trading","algorithmic trading","high-frequency","flash crash",
    "investment","robo-advisor","portfolio",
    "payment","transaction","insurance","underwriting",
    "risk assessment","wealth management","hedge fund",
    "stock","equity","derivative","securities"
]

def is_financial(row):
    text = (str(row.get("title","")) + " " + str(row.get("description",""))).lower()
    return any(kw in text for kw in FINANCIAL_KEYWORDS)

df_finance = df_original[df_original.apply(is_financial, axis=1)].copy()

print("="*80); print("🎯 FILTRAGEM — SETOR FINANCEIRO"); print("="*80)
print(f"📊 Originais  : {len(df_original):,}")
print(f"💰 Financeiros: {len(df_finance):,}")
print(f"📈 Percentual : {len(df_finance)/len(df_original)*100:.2f}%")
if len(df_finance) < 20:
    print("⚠️  Amostra muito pequena — verifique keywords e dataset.")
else:
    print("\n✅ Filtragem concluída com sucesso!")

## 6️⃣ Limpeza e padronização

**O que faz**: Renomeia colunas para `snake_case`, converte datas, cria texto unificado, remove duplicatas.

**Por que é importante**: Padronização garante consistência com os notebooks seguintes.

In [ ]:
rename_map = {
    "date":"occurred_date", "date_published":"occurred_date",
    "description":"summary",
    "Alleged deployer of AI system":"deployer",
    "Alleged developer of AI system":"developer",
    "Alleged harmed or nearly harmed parties":"harmed_parties"
}
for old,new in rename_map.items():
    if old in df_finance.columns and new not in df_finance.columns:
        df_finance = df_finance.rename(columns={old: new})

if "occurred_date" in df_finance.columns:
    df_finance["occurred_date"] = pd.to_datetime(df_finance["occurred_date"], errors="coerce")
    df_finance["year"] = df_finance["occurred_date"].dt.year
else:
    df_finance["year"] = np.nan

df_finance["title"]   = df_finance.get("title",   pd.Series(dtype=str)).fillna("")
df_finance["summary"] = df_finance.get("summary", pd.Series(dtype=str)).fillna("")
df_finance["text"]    = (df_finance["title"] + " " + df_finance["summary"]).str.lower().str.strip()

id_col = next((c for c in ["incident_id","report_number"] if c in df_finance.columns), None)
if id_col and id_col != "incident_id":
    df_finance = df_finance.rename(columns={id_col: "incident_id"})
if "incident_id" not in df_finance.columns:
    df_finance["incident_id"] = range(1, len(df_finance)+1)

before = len(df_finance)
df_finance = df_finance.drop_duplicates(subset=["incident_id"], keep="first")

print("="*80); print("✅ LIMPEZA CONCLUÍDA"); print("="*80)
print(f"📊 Registros finais    : {len(df_finance):,}")
print(f"🔄 Duplicatas removidas: {before - len(df_finance)}")
if df_finance["year"].notna().any():
    print(f"📅 Período             : {int(df_finance['year'].min())} – {int(df_finance['year'].max())}")
display(df_finance[["incident_id","occurred_date","year","title"]].head(3))

## 7️⃣ Engenharia de Features — 8 Variáveis Derivadas

**O que faz**: Cria variáveis categóricas a partir do texto dos incidentes por correspondência de palavras-chave.

**Bugs corrigidos:**
- ⚠️ `severity_level`: 29/32 incidentes classificados como "low" — keywords muito restritas. Flash Crash, MiDAS, deepfakes financeiros eram "low"
- ⚠️ `application_type`: 14/32 como "other_finance" — keywords de detecção insuficientes
- ⚠️ Ordem das condições em `classify_application_type`: "fraud" interceptava casos de deepfake que deveriam ser "other"

**Lógica de classificação**: cada função retorna a primeira categoria cujas keywords aparecem no texto. Mais específico primeiro, mais geral por último.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FEATURE ENGINEERING — 8 variáveis derivadas por classificação por keywords
# ══════════════════════════════════════════════════════════════════════════════

def classify_application_type(text: str) -> str:
    """
    Classifica o tipo de aplicação de IA financeira envolvida no incidente.

    Retorna a primeira categoria cujas keywords aparecem no texto.
    Mais específico primeiro → mais genérico por último.
    """
    t = str(text).lower()

    if any(k in t for k in [
        "credit scor", "loan approv", "loan denial", "lending",
        "mortgage", "credit decision", "credit card approv",
        "credit limit", "line of credit",
    ]):
        return "credit_scoring"

    elif any(k in t for k in [
        "robo-advis", "robo advis", "robo advisor",
        "investment advis", "portfolio manag", "wealth manag",
        "asset manag", "automated invest",
    ]):
        return "robo_advisor"

    elif any(k in t for k in [
        "flash crash", "high-frequency", "algorithmic trad",
        "market manipul", "stock trad", "quantitative trad",
        "trading algorithm", "hft",
    ]):
        return "algorithmic_trading"

    elif any(k in t for k in [
        "underwriting", "insurance claim", "insurance pric",
        "actuarial", "risk underwrit",
    ]):
        return "risk_assessment"

    elif any(k in t for k in [
        "recidiv", "risk assessment", "risk scor",
        "risk classif", "risk predict",
    ]):
        return "risk_assessment"

    elif any(k in t for k in [
        "chatbot", "customer service bot", "virtual assistant",
        "call center ai", "automated customer",
    ]):
        return "customer_service"

    elif any(k in t for k in [
        "fraud detect", "aml", "anti-money launder",
        "money launder", "suspicious transaction",
        "fraud algorithm", "fraud system", "fraud claim",
        "unemployment benefit", "tax fraud", "benefits fraud",
        "deepfake", "voice clone", "voice spoof",
        "identity fraud", "account takeover",
    ]):
        # Fraude inclui detecção de fraude E os próprios golpes com IA
        return "fraud_detection"

    elif any(k in t for k in [
        "payment", "transaction", "bank transfer",
        "wire transfer", "fintech",
    ]):
        return "other_finance"

    else:
        return "other_finance"


def classify_incident_type(text: str) -> str:
    """
    Classifica a natureza do incidente de IA.

    Prioridade: bias > market_disruption > data_breach >
                regulatory_violation > operational_failure > other
    """
    t = str(text).lower()

    if any(k in t for k in [
        "bias", "biased", "discriminat", "unfair",
        "gender", "racial", "race", "disproportion",
        "prejudice", "stereotype", "disparate impact",
        "demographic", "minority", "unequal",
    ]):
        return "algorithmic_bias"

    elif any(k in t for k in [
        "flash crash", "market disruption", "market crash",
        "volatility spike", "circuit breaker", "trading halt",
        "market manipul", "pump and dump",
    ]):
        return "market_disruption"

    elif any(k in t for k in [
        "data breach", "data leak", "hack", "cyber",
        "data expos", "privacy violat", "personal data",
        "gdpr", "stolen data", "unauthorized access",
    ]):
        return "data_breach"

    elif any(k in t for k in [
        "regulat", "violation", "sanction", "compliance",
        "illegal", "lawsuit", "sued", "litigation",
        "class action", "ftc", "sec investigation",
    ]):
        return "regulatory_violation"

    elif any(k in t for k in [
        "crash", "failure", "outage", "malfunction",
        "error", "bug", "glitch", "false positive",
        "false negative", "incorrect", "wrong",
        "mistaken", "inaccurate", "hallucin",
    ]):
        return "operational_failure"

    else:
        return "other"


def classify_customer_segment(text: str) -> str:
    """Identifica qual segmento de cliente foi afetado pelo incidente."""
    t = str(text).lower()

    if any(k in t for k in [
        "underserved", "minority", "low-income", "low income",
        "vulnerable", "disadvantaged", "marginalized",
        "welfare", "benefits recipient", "unemployed",
        "working class", "unbanked",
    ]):
        return "underserved"   # mais específico primeiro

    elif any(k in t for k in [
        "small business", "sme", "small and medium",
        "startup", "entrepreneur", "self-employed",
    ]):
        return "sme"

    elif any(k in t for k in [
        "corporate", "enterprise", "large compan",
        "institution", "hedge fund", "pension fund",
        "investment bank",
    ]):
        return "corporate"

    elif any(k in t for k in [
        "retail", "consumer", "individual", "personal",
        "customer", "client", "user", "resident",
        "applicant", "employee", "worker", "driver",
    ]):
        return "retail"

    else:
        return "general"


def classify_severity(text: str) -> str:
    """
    Classifica a severidade do incidente em 4 níveis.

    BUG ANTERIOR: 29/32 incidentes classificados como 'low' porque:
    - Flash Crash (trilhões perdidos) → 'low'
    - MiDAS (40.000 falsas acusações) → 'low'
    - Deepfakes financeiros → 'low'

    CORREÇÃO: keywords expandidas e ponderadas. Palavras que indicam
    impacto sistêmico, perdas massivas ou ação judicial → high/critical.
    """
    t = str(text).lower()

    # ── CRITICAL: impacto sistêmico, perdas massivas, falência ───────────────
    if any(k in t for k in [
        "bankrupt", "systemic", "shutdown", "catastrophic",
        "trillion", "billion dollar", "billions",
        "market crash", "flash crash",          # Flash Crash: $1T perdido
        "thousands of people", "tens of thousands",
        "class action", "massive fine",
        "suspended operation", "shut down",
    ]):
        return "critical"

    # ── HIGH: investigação formal, perdas significativas, processo judicial ───
    elif any(k in t for k in [
        "investigation", "lawsuit", "sued", "litigation",
        "penalty", "settlement", "million dollar",
        "million fine", "significant loss", "major loss",
        "wrongful", "false accusation", "falsely accused",
        "regulatory action", "ftc", "sec ",
        "civil rights", "discrimination lawsuit",
        "deepfake", "voice clone",  # fraudes com IA → alto impacto
    ]):
        return "high"

    # ── MEDIUM: reclamações formais, revisões, impacto moderado documentado ──
    elif any(k in t for k in [
        "complaint", "fine", "sanction",
        "audit", "review", "concern", "criticized",
        "allegedly", "reported", "claimed", "accused",
        "error rate", "inaccurate", "unreliable",
        "bias report", "discrimination report",
    ]):
        return "medium"

    # ── LOW: impacto limitado, experimental, sem consequências documentadas ──
    else:
        return "low"


def extract_governance_flags(text: str) -> dict:
    """
    Extrai 4 flags binárias de resposta de governança/regulatória.

    Cada flag indica se aquele tipo de resposta ocorreu (1) ou não (0).
    """
    t = str(text).lower()

    return {
        "regulatory_investigation": int(any(k in t for k in [
            "investigation", "probe", "inquiry",
            "sec ", "ftc ", "regulator", "regulatory action",
            "department of justice", "doj", "oversight",
        ])),
        "fine_imposed": int(any(k in t for k in [
            "fine", "penalty", "sanction", "settlement",
            "paid", "million fine", "ordered to pay",
        ])),
        "policy_change": int(any(k in t for k in [
            "policy change", "policy update", "suspended",
            "discontinued", "halted", "stopped using",
            "removed", "pulled", "banned", "prohibited",
        ])),
        "third_party_audit": int(any(k in t for k in [
            "audit", "independent review", "third-party",
            "external review", "third party audit",
            "independent assessment",
        ])),
    }


# ── Aplicar feature engineering ───────────────────────────────────────────────
print("⚙️  Aplicando classificações...")

df_finance["application_type"] = df_finance["text"].apply(classify_application_type)
df_finance["incident_type"]    = df_finance["text"].apply(classify_incident_type)
df_finance["customer_segment"] = df_finance["text"].apply(classify_customer_segment)
df_finance["severity_level"]   = df_finance["text"].apply(classify_severity)

gov_df     = df_finance["text"].apply(extract_governance_flags).apply(pd.Series)
df_finance = pd.concat([df_finance, gov_df], axis=1)

# ── Relatório de distribuição ─────────────────────────────────────────────────
print("=" * 80)
print("✅ 8 VARIÁVEIS DERIVADAS CRIADAS")
print("=" * 80)

for var in ["application_type", "incident_type", "customer_segment", "severity_level"]:
    counts = df_finance[var].value_counts()
    print(f"\n{var.upper()}:")
    for cat, n in counts.items():
        pct = n / len(df_finance) * 100
        bar = "█" * int(pct / 3)
        print(f"  {cat:30s} {n:4d} ({pct:5.1f}%)  {bar}")

print("\nFLAGS DE GOVERNANÇA:")
for f in ["regulatory_investigation", "fine_imposed", "policy_change", "third_party_audit"]:
    n   = int(df_finance[f].sum())
    pct = n / len(df_finance) * 100
    print(f"  {f:30s} {n:4d} ({pct:5.1f}%)")

## 8️⃣ Visualização das variáveis derivadas

**O que faz**: Painel 2×2 com a distribuição de cada variável categórica.

**Caso de uso**: Validação visual antes de avançar para análise estatística.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("📊 Distribuição das Variáveis Derivadas — Incidentes Financeiros de IA",
             fontsize=15, fontweight="bold", y=0.998)

configs = [
    (axes[0,0], "application_type", "steelblue",     "Tipo de Aplicação de IA"),
    (axes[0,1], "incident_type",    "coral",          "Tipo de Incidente"),
    (axes[1,0], "customer_segment", "mediumseagreen", "Segmento de Cliente Afetado"),
    (axes[1,1], "severity_level",   None,             "Nível de Severidade"),
]

for ax, col, color, title in configs:
    counts = df_finance[col].value_counts()
    if col == "severity_level":
        order  = ["low","medium","high","critical"]
        counts = counts.reindex([s for s in order if s in counts.index], fill_value=0)
        colors = ["#90EE90","#FFD700","#FF8C00","#DC143C"][:len(counts)]
        bars = ax.bar(range(len(counts)), counts.values, color=colors)
        ax.set_xticks(range(len(counts))); ax.set_xticklabels(counts.index)
        for b in bars:
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.2,
                    str(int(b.get_height())), ha="center", fontsize=9)
        ax.grid(axis="y", alpha=0.3)
    else:
        ax.barh(counts.index, counts.values, color=color)
        for i,v in enumerate(counts.values):
            ax.text(v+0.1, i, str(v), va="center", fontsize=9)
        ax.grid(axis="x", alpha=0.3)
    ax.set_title(title, fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("distribuicao_variaveis.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Visualizações geradas.")

## 9️⃣ Salvamento do CSV processado

**O que faz**: Persiste o dataset limpo e enriquecido como `incidents_finance_filtered.csv`.

**Caso de uso**: Ponto de entrada dos Notebooks 2, 3 e 4.

In [ ]:
COLS_TO_SAVE = [
    "incident_id","occurred_date","title","summary","year","text",
    "application_type","incident_type","customer_segment","severity_level",
    "regulatory_investigation","fine_imposed","policy_change","third_party_audit"
]
available = [c for c in COLS_TO_SAVE if c in df_finance.columns]
df_final  = df_finance[available].copy()
df_final.to_csv("incidents_finance_filtered.csv", index=False)

print("="*80); print("💾 CSV SALVO: incidents_finance_filtered.csv"); print("="*80)
print(f"📊 Registros: {len(df_final):,}  |  📋 Colunas: {len(df_final.columns)}")
print(f"Colunas: {', '.join(available)}")

## 🔟 Criação do banco de dados SQLite

**O que faz**: Cria e popula `ai_finance_incidents.db` com 3 tabelas relacionadas por chaves estrangeiras.

| Tabela | Chave Primária | Chave Estrangeira | Descrição |
|---|---|---|---|
| `incidents` | `incident_id` | — | Tabela principal |
| `financial_impacts` | `impact_id` | `incident_id` | Impacto financeiro |
| `regulatory_responses` | `response_id` | `incident_id` | Resposta de governança |

**Caso de uso**: Requisito do briefing (≥ 3 tabelas) e base da API RESTful.

In [ ]:
DB_FILE = "ai_finance_incidents.db"
conn    = sqlite3.connect(DB_FILE)
cur     = conn.cursor()

for tbl in ("regulatory_responses","financial_impacts","incidents"):
    cur.execute(f"DROP TABLE IF EXISTS {tbl}")

cur.execute("""CREATE TABLE incidents (
    incident_id INTEGER PRIMARY KEY, title TEXT NOT NULL, summary TEXT,
    occurred_date DATE, year INTEGER, application_type TEXT,
    customer_segment TEXT, incident_type TEXT, severity_level TEXT, text TEXT)""")

cur.execute("""CREATE TABLE financial_impacts (
    impact_id INTEGER PRIMARY KEY AUTOINCREMENT, incident_id INTEGER NOT NULL,
    severity_level TEXT, estimated_loss TEXT, impact_description TEXT,
    FOREIGN KEY (incident_id) REFERENCES incidents(incident_id))""")

cur.execute("""CREATE TABLE regulatory_responses (
    response_id INTEGER PRIMARY KEY AUTOINCREMENT, incident_id INTEGER NOT NULL,
    regulatory_investigation INTEGER DEFAULT 0, fine_imposed INTEGER DEFAULT 0,
    policy_change INTEGER DEFAULT 0, third_party_audit INTEGER DEFAULT 0,
    response_description TEXT,
    FOREIGN KEY (incident_id) REFERENCES incidents(incident_id))""")

# Popular incidents
inc_cols = ["incident_id","title","summary","occurred_date","year",
            "application_type","customer_segment","incident_type","severity_level","text"]
inc      = df_final[[c for c in inc_cols if c in df_final.columns]].copy()
if "occurred_date" in inc.columns:
    inc["occurred_date"] = inc["occurred_date"].astype(str)
inc.to_sql("incidents", conn, if_exists="append", index=False)

# Popular financial_impacts
fi = df_final[["incident_id","severity_level"]].copy()
fi["estimated_loss"]     = "Not reported"
fi["impact_description"] = df_final["summary"].fillna("").str[:200]
fi.to_sql("financial_impacts", conn, if_exists="append", index=False)

# Popular regulatory_responses
rr_cols = [c for c in ["incident_id","regulatory_investigation",
                        "fine_imposed","policy_change","third_party_audit"] if c in df_final.columns]
rr = df_final[rr_cols].copy()
rr["response_description"] = "Extraído do texto do incidente"
rr.to_sql("regulatory_responses", conn, if_exists="append", index=False)

conn.commit()
print("="*80); print(f"🗄️  BANCO: {DB_FILE}"); print("="*80)
cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
for (tbl,) in cur.fetchall():
    cur.execute(f"SELECT COUNT(*) FROM {tbl}")
    print(f"  ✓ {tbl:30s}: {cur.fetchone()[0]:,} registros")

print("\n🔗 Exemplo JOIN:")
q = """SELECT i.incident_id, i.severity_level, r.regulatory_investigation
FROM incidents i JOIN regulatory_responses r USING(incident_id)
WHERE i.severity_level IN ('high','critical') LIMIT 5"""
display(pd.read_sql_query(q, conn))
conn.close()

In [ ]:
print("="*80); print("🎉 NOTEBOOK 1 CONCLUÍDO!"); print("="*80)
print(f"📊 Incidentes processados: {len(df_final):,}")
if df_final["year"].notna().any():
    print(f"📅 Período: {int(df_final['year'].min())} – {int(df_final['year'].max())}")
print("\n📁 Arquivos gerados:")
print("  ✓ incidents_finance_filtered.csv")
print("  ✓ ai_finance_incidents.db")
print("  ✓ distribuicao_variaveis.png")
print("\n➡️  Execute o Notebook 2: Análise Estatística e Testes de Hipóteses")